In [1]:
df = spark.read.csv('Files/Bronze/sales_data.csv', header=True, inferSchema=True)

df.show(5)

StatementMeta(, 5d60beab-3716-4ba4-9a60-eaa1a727da7c, 3, Finished, Available, Finished, False)

+----------+----------+-----------+-----+--------+-------+
|      date|   product|   category|price|quantity|revenue|
+----------+----------+-----------+-----+--------+-------+
|2022-01-01|Smartphone|Electronics|  600|      10|   6000|
|2022-01-01|    Laptop|Electronics| 1200|       5|   6000|
|2022-01-02|   T-Shirt|   Clothing|   20|      50|   1000|
|2022-01-03|Headphones|Electronics|  100|      20|   2000|
|2022-01-04|   T-Shirt|   Clothing|   20|      25|    500|
+----------+----------+-----------+-----+--------+-------+
only showing top 5 rows



In [2]:
# Remove null values
df_clean = df.dropna()

# Remove duplicate records
df_clean = df_clean.dropDuplicates()

# Preview cleaned data
df_clean.show(5)

StatementMeta(, 5d60beab-3716-4ba4-9a60-eaa1a727da7c, 4, Finished, Available, Finished, False)

+----------+----------+-----------+-----+--------+-------+
|      date|   product|   category|price|quantity|revenue|
+----------+----------+-----------+-----+--------+-------+
|2022-04-20|   T-Shirt|   Clothing|   20|      20|    400|
|2022-09-19|  Backpack|       Bags|   50|      15|    750|
|2022-01-01|    Laptop|Electronics| 1200|       5|   6000|
|2022-01-28|   Speaker|Electronics|   80|       8|    640|
|2022-06-27|Smartphone|Electronics|  600|       9|   5400|
+----------+----------+-----------+-----+--------+-------+
only showing top 5 rows



In [3]:
from pyspark.sql.functions import col

df_clean.printSchema()

StatementMeta(, 5d60beab-3716-4ba4-9a60-eaa1a727da7c, 5, Finished, Available, Finished, False)

root
 |-- date: date (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- revenue: integer (nullable = true)



In [4]:
from pyspark.sql.functions import col, upper

df_clean = df_clean \
    .withColumn("product", upper(col("product"))) \
    .withColumn("category", upper(col("category")))

StatementMeta(, 5d60beab-3716-4ba4-9a60-eaa1a727da7c, 6, Finished, Available, Finished, False)

In [5]:
df_clean.write.mode("overwrite").saveAsTable("silver_sales")

StatementMeta(, 5d60beab-3716-4ba4-9a60-eaa1a727da7c, 7, Finished, Available, Finished, False)

In [6]:
df_clean = df_clean.withColumn("total_value", col("price") * col("quantity"))

StatementMeta(, 5d60beab-3716-4ba4-9a60-eaa1a727da7c, 8, Finished, Available, Finished, False)

In [7]:
from pyspark.sql.functions import sum, col

gold_df = df_clean.groupBy("category").agg(
    sum("revenue").alias("total_revenue"),
    sum("quantity").alias("total_quantity")
)

gold_df.show()

StatementMeta(, 5d60beab-3716-4ba4-9a60-eaa1a727da7c, 9, Finished, Available, Finished, False)

+-----------+-------------+--------------+
|   category|total_revenue|total_quantity|
+-----------+-------------+--------------+
|      SHOES|        20640|           258|
|ACCESSORIES|       101400|           902|
|   CLOHTING|         1200|            30|
|       BAGS|        19500|           390|
|    SHOESES|          960|            12|
|ELECTRONICS|       509480|          1439|
|   CLOTHING|        93150|          2221|
|       BGAS|          900|            18|
+-----------+-------------+--------------+



In [8]:
gold_df = gold_df.orderBy(col("total_revenue").desc())

StatementMeta(, 5d60beab-3716-4ba4-9a60-eaa1a727da7c, 10, Finished, Available, Finished, False)

In [9]:
gold_df.write.mode("overwrite").saveAsTable("gold_sales")

StatementMeta(, 5d60beab-3716-4ba4-9a60-eaa1a727da7c, 11, Finished, Available, Finished, False)

In [10]:
from pyspark.sql.functions import when, col

df_clean = df_clean.withColumn(
    "category",
    when(col("category") == "bgas", "BAGS")
    .when(col("category") == "clohting", "CLOTHING")
    .when(col("category") == "shoeses", "SHOES")
    .otherwise(col("category"))
)

StatementMeta(, 5d60beab-3716-4ba4-9a60-eaa1a727da7c, 12, Finished, Available, Finished, False)

In [11]:
df_clean.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver_sales")

StatementMeta(, 5d60beab-3716-4ba4-9a60-eaa1a727da7c, 13, Finished, Available, Finished, False)

In [12]:
from pyspark.sql.functions import sum

gold_df = df_clean.groupBy("category").agg(
    sum("revenue").alias("total_revenue"),
    sum("quantity").alias("total_quantity")
)

gold_df.write.mode("overwrite").saveAsTable("gold_sales")

StatementMeta(, 5d60beab-3716-4ba4-9a60-eaa1a727da7c, 14, Finished, Available, Finished, False)

In [13]:
df = spark.read.csv('Files/Bronze/sales_data.csv', header=True, inferSchema=True)

df_clean = df.dropna().dropDuplicates()

StatementMeta(, 5d60beab-3716-4ba4-9a60-eaa1a727da7c, 15, Finished, Available, Finished, False)

In [14]:
from pyspark.sql.functions import when, col, upper

df_clean = df_clean.withColumn("category", upper(col("category")))

df_clean = df_clean.withColumn(
    "category",
    when(col("category") == "BGAS", "BAGS")
    .when(col("category") == "CLOHTING", "CLOTHING")
    .when(col("category") == "SHOESES", "SHOES")
    .otherwise(col("category"))
)

StatementMeta(, 5d60beab-3716-4ba4-9a60-eaa1a727da7c, 16, Finished, Available, Finished, False)

In [15]:
df_clean.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver_sales")

StatementMeta(, 5d60beab-3716-4ba4-9a60-eaa1a727da7c, 17, Finished, Available, Finished, False)

In [16]:
from pyspark.sql.functions import sum

gold_df = df_clean.groupBy("category").agg(
    sum("revenue").alias("total_revenue"),
    sum("quantity").alias("total_quantity")
)

gold_df.write.mode("overwrite").saveAsTable("gold_sales")

StatementMeta(, 5d60beab-3716-4ba4-9a60-eaa1a727da7c, 18, Finished, Available, Finished, False)

In [17]:
from pyspark.sql.functions import col, to_date

df = spark.read.csv('Files/Bronze/sales_data.csv', header=True, inferSchema=True)

df = df.withColumn("date", to_date(col("date"), "M/d/yyyy"))

historical_df = df.filter(col("date") <= "2022-03-31")

historical_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze_sales")

StatementMeta(, 5d60beab-3716-4ba4-9a60-eaa1a727da7c, 19, Finished, Available, Finished, False)

In [18]:
from pyspark.sql.functions import col

new_data_df = df.filter(
    (col("date") > "2022-03-31") &
    (col("date") <= "2022-06-30")
)


from delta.tables import DeltaTable

target = DeltaTable.forName(spark, "bronze_sales")

(target.alias("old")
 .merge(
     new_data_df.alias("new"),
     "old.date = new.date AND old.product = new.product AND old.category = new.category"
 )
 .whenMatchedUpdateAll()
 .whenNotMatchedInsertAll()
 .execute())

StatementMeta(, 5d60beab-3716-4ba4-9a60-eaa1a727da7c, 20, Finished, Available, Finished, False)

In [19]:
from pyspark.sql.functions import col

new_data_df = df.filter(
    (col("date") > "2022-06-30") &
    (col("date") <= "2022-09-30")
)


from delta.tables import DeltaTable

target = DeltaTable.forName(spark, "bronze_sales")

(target.alias("old")
 .merge(
     new_data_df.alias("new"),
     "old.date = new.date AND old.product = new.product AND old.category = new.category"
 )
 .whenMatchedUpdateAll()
 .whenNotMatchedInsertAll()
 .execute())

StatementMeta(, 5d60beab-3716-4ba4-9a60-eaa1a727da7c, 21, Finished, Available, Finished, False)

In [20]:
from pyspark.sql.functions import col

new_data_df = df.filter(
    (col("date") > "2022-09-30") &
    (col("date") <= "2022-12-31")
)


from delta.tables import DeltaTable

target = DeltaTable.forName(spark, "bronze_sales")

(target.alias("old")
 .merge(
     new_data_df.alias("new"),
     "old.date = new.date AND old.product = new.product AND old.category = new.category"
 )
 .whenMatchedUpdateAll()
 .whenNotMatchedInsertAll()
 .execute())

StatementMeta(, 5d60beab-3716-4ba4-9a60-eaa1a727da7c, 22, Finished, Available, Finished, False)

In [21]:
df_clean = spark.table("bronze_sales")

StatementMeta(, 5d60beab-3716-4ba4-9a60-eaa1a727da7c, 23, Finished, Available, Finished, False)

In [22]:
df_clean = df_clean.dropna()

StatementMeta(, 5d60beab-3716-4ba4-9a60-eaa1a727da7c, 24, Finished, Available, Finished, False)

In [23]:
df_clean = df_clean.dropDuplicates()

StatementMeta(, 5d60beab-3716-4ba4-9a60-eaa1a727da7c, 25, Finished, Available, Finished, False)

In [24]:
from pyspark.sql.functions import col, upper

df_clean = df_clean.withColumn("category", upper(col("category")))

StatementMeta(, 5d60beab-3716-4ba4-9a60-eaa1a727da7c, 26, Finished, Available, Finished, False)

In [25]:
from pyspark.sql.functions import when

df_clean = df_clean.withColumn(
    "category",
    when(col("category") == "BGAS", "BAGS")
    .when(col("category") == "CLOHTING", "CLOTHING")
    .when(col("category") == "SHOESES", "SHOES")
    .otherwise(col("category"))
)

StatementMeta(, 5d60beab-3716-4ba4-9a60-eaa1a727da7c, 27, Finished, Available, Finished, False)

In [26]:
df_clean.select("category").distinct().show()

StatementMeta(, 5d60beab-3716-4ba4-9a60-eaa1a727da7c, 28, Finished, Available, Finished, False)

+-----------+
|   category|
+-----------+
|      SHOES|
|ACCESSORIES|
|       BAGS|
|ELECTRONICS|
|   CLOTHING|
+-----------+



In [27]:
df_clean = df_clean.withColumn("revenue", col("price") * col("quantity"))

StatementMeta(, 5d60beab-3716-4ba4-9a60-eaa1a727da7c, 29, Finished, Available, Finished, False)

In [28]:
df_clean.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("category") \
    .saveAsTable("silver_sales")

StatementMeta(, 5d60beab-3716-4ba4-9a60-eaa1a727da7c, 30, Finished, Available, Finished, False)

In [29]:
spark.table("silver_sales").show()

StatementMeta(, 5d60beab-3716-4ba4-9a60-eaa1a727da7c, 31, Finished, Available, Finished, False)

+----------+----------+-----------+-----+--------+-------+
|      date|   product|   category|price|quantity|revenue|
+----------+----------+-----------+-----+--------+-------+
|2022-02-05|     Watch|ACCESSORIES|  150|       8|   1200|
|2022-02-28|     Watch|ACCESSORIES|  150|       5|    750|
|2022-03-05|Smartwatch|ACCESSORIES|  200|      12|   2400|
|2022-02-25|Smartwatch|ACCESSORIES|  200|      10|   2000|
|2022-02-14|    Wallet|ACCESSORIES|   30|      30|    900|
|2022-01-16|Smartwatch|ACCESSORIES|  200|       5|   1000|
|2022-01-05|     Watch|ACCESSORIES|  150|      10|   1500|
|2022-01-08|Smartwatch|ACCESSORIES|  200|      12|   2400|
|2022-02-16|Smartwatch|ACCESSORIES|  200|       7|   1400|
|2022-01-25|Smartwatch|ACCESSORIES|  200|       8|   1600|
|2022-01-14|    Wallet|ACCESSORIES|   30|      40|   1200|
|2022-03-25|Smartwatch|ACCESSORIES|  200|       8|   1600|
|2022-03-14|    Wallet|ACCESSORIES|   30|      40|   1200|
|2022-03-15|Smartwatch|ACCESSORIES|  200|       5|   100

In [30]:
spark.sql("SHOW PARTITIONS silver_sales").show()

StatementMeta(, 5d60beab-3716-4ba4-9a60-eaa1a727da7c, 32, Finished, Available, Finished, False)

+--------------------+
|           partition|
+--------------------+
|category=ACCESSORIES|
|       category=BAGS|
|   category=CLOTHING|
|category=ELECTRONICS|
|      category=SHOES|
+--------------------+



In [31]:
df_gold = spark.table("silver_sales")

StatementMeta(, 5d60beab-3716-4ba4-9a60-eaa1a727da7c, 33, Finished, Available, Finished, False)

In [32]:
from pyspark.sql.functions import sum

gold_df = df_gold.groupBy("category").agg(
    sum("revenue").alias("total_revenue"),
    sum("quantity").alias("total_quantity")
)

StatementMeta(, 5d60beab-3716-4ba4-9a60-eaa1a727da7c, 34, Finished, Available, Finished, False)

In [33]:
gold_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_sales")

StatementMeta(, 5d60beab-3716-4ba4-9a60-eaa1a727da7c, 35, Finished, Available, Finished, False)

In [34]:
spark.table("gold_sales").show()

StatementMeta(, 5d60beab-3716-4ba4-9a60-eaa1a727da7c, 36, Finished, Available, Finished, False)

+-----------+-------------+--------------+
|   category|total_revenue|total_quantity|
+-----------+-------------+--------------+
|ELECTRONICS|       509480|          1439|
|   CLOTHING|        94350|          2251|
|ACCESSORIES|       101400|           902|
|       BAGS|        20400|           408|
|      SHOES|        21600|           270|
+-----------+-------------+--------------+



In [35]:
df_test = spark.table("silver_sales")

df_test = df_test.withColumn("revenue", col("revenue") + 10)

df_test.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("category") \
    .saveAsTable("silver_sales")

StatementMeta(, 5d60beab-3716-4ba4-9a60-eaa1a727da7c, 37, Finished, Available, Finished, False)

In [36]:
gold_df = df_test.groupBy("category").agg(
    sum("revenue").alias("total_revenue"),
    sum("quantity").alias("total_quantity")
)

gold_df.write.mode("overwrite").saveAsTable("gold_sales")

StatementMeta(, 5d60beab-3716-4ba4-9a60-eaa1a727da7c, 38, Finished, Available, Finished, False)

In [37]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

# Read Silver Layer
df = spark.table("silver_sales")

# Window specification
window_spec = Window.orderBy("date")

# Create Fact Table with Sequential IDs
fact_sales = df.withColumn(
    "sale_id",
    row_number().over(window_spec)
).select(
    "sale_id",
    "date",
    "category",
    "revenue",
    "quantity"
)

# Save Fact Table
fact_sales.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("fact_sales")

StatementMeta(, 5d60beab-3716-4ba4-9a60-eaa1a727da7c, 39, Finished, Available, Finished, False)

In [38]:
spark.sql("SELECT * FROM fact_sales LIMIT 5").show()

StatementMeta(, 5d60beab-3716-4ba4-9a60-eaa1a727da7c, 40, Finished, Available, Finished, False)

+-------+----------+-----------+-------+--------+
|sale_id|      date|   category|revenue|quantity|
+-------+----------+-----------+-------+--------+
|      1|2022-01-01|ELECTRONICS|   6010|       5|
|      2|2022-01-01|ELECTRONICS|   6010|      10|
|      3|2022-01-02|   CLOTHING|   1010|      50|
|      4|2022-01-03|ELECTRONICS|   2010|      20|
|      5|2022-01-04|   CLOTHING|    510|      25|
+-------+----------+-----------+-------+--------+



In [39]:
from pyspark.sql.functions import dense_rank
from pyspark.sql.window import Window

# Read Silver Layer
df = spark.table("silver_sales")

# Window for IDs
window_spec = Window.orderBy("category")

# Create Product Dimension
dim_product = df.select("category").distinct() \
    .withColumn(
        "product_id",
        dense_rank().over(window_spec)
    ) \
    .select(
        "product_id",
        "category"
    )

# Save Dimension Table
dim_product.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("dim_product")

StatementMeta(, 5d60beab-3716-4ba4-9a60-eaa1a727da7c, 41, Finished, Available, Finished, False)

In [40]:
from pyspark.sql.functions import \
    year, month, quarter, dayofmonth
from pyspark.sql.window import Window
from pyspark.sql.functions import dense_rank

# Read Silver Layer
df = spark.table("silver_sales")

# Create Date Dimension
dim_date = df.select("date").distinct() \
    .withColumn("day", dayofmonth("date")) \
    .withColumn("month", month("date")) \
    .withColumn("quarter", quarter("date")) \
    .withColumn("year", year("date"))

# Generate Date IDs
window_spec = Window.orderBy("date")

dim_date = dim_date.withColumn(
    "date_id",
    dense_rank().over(window_spec)
).select(
    "date_id",
    "date",
    "day",
    "month",
    "quarter",
    "year"
)

# Save Date Dimension
dim_date.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("dim_date")

StatementMeta(, 5d60beab-3716-4ba4-9a60-eaa1a727da7c, 42, Finished, Available, Finished, False)

In [41]:
# Read tables
silver_df = spark.table("silver_sales")
dim_product = spark.table("dim_product")
dim_date = spark.table("dim_date")

# Join with Product Dimension
fact_sales = silver_df.join(
    dim_product,
    on="category",
    how="left"
)

# Join with Date Dimension
fact_sales = fact_sales.join(
    dim_date,
    on="date",
    how="left"
)

# Select final fact table columns
fact_sales = fact_sales.select(
    "product_id",
    "date_id",
    "revenue",
    "quantity"
)

# Save final fact table
fact_sales.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("fact_sales")

StatementMeta(, 5d60beab-3716-4ba4-9a60-eaa1a727da7c, 43, Finished, Available, Finished, False)